Setup Kaggle n Git

In [31]:
import os
import warnings
warnings.filterwarnings("ignore")

DATA_PATH = "/kaggle/input/datasets/williamscott701/memotion-dataset-7k/memotion_dataset_7k"

print(os.listdir(DATA_PATH))

['reference.csv', 'labels_pd_pickle', 'labels.xlsx', 'reference.xlsx', 'images', 'labels.csv', 'reference_df_pickle']


In [32]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
GITHUB_TOKEN = user_secrets.get_secret("Hate Meme Repo Access Token")
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

In [33]:
!git clone https://{token}@github.com/Parthivendra/multimodal-hate-meme-detection.git
%cd multimodal-hate-meme-detection

Cloning into 'multimodal-hate-meme-detection'...
remote: Enumerating objects: 77, done.
remote: Counting objects: 100% (77/77), done.
remote: Compressing objects: 100% (54/54), done.
remote: Total 77 (delta 47), reused 47 (delta 20), pack-reused 0 (from 0)
Receiving objects: 100% (77/77), 11.18 KiB | 11.18 MiB/s, done.
Resolving deltas: 100% (47/47), done.
/kaggle/working/multimodal-hate-meme-detection/multimodal-hate-meme-detection


In [63]:
!git pull

remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 2), reused 4 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 553 bytes | 553.00 KiB/s, done.
From https://github.com/Parthivendra/multimodal-hate-meme-detection
   de35dae..f1c1d4c  main       -> origin/main
Updating de35dae..f1c1d4c
Fast-forward
 src/model_image.py | 16 ++++++++++++++++
 1 file changed, 16 insertions(+)
 create mode 100644 src/model_image.py


In [35]:
import sys
sys.path.append("/kaggle/working")
sys.path.append("/kaggle/working/multimodal-hate-meme-detection")

In [65]:
# reload custom modules in case of remote update via git or otherwise
import importlib
import src.dataset
import src.model_text
import src.model_image
import src.train
importlib.reload(src.dataset)
importlib.reload(src.model_text)
importlib.reload(src.model_image)
importlib.reload(src.train)

<module 'src.train' from '/kaggle/working/multimodal-hate-meme-detection/multimodal-hate-meme-detection/src/train.py'>

Code

In [37]:
from huggingface_hub import login
login(HF_TOKEN)

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [38]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

In [39]:
# import and validate dataset
import pandas as pd
import os

df = pd.read_csv(f"{DATA_PATH}/labels.csv", index_col=0)

# check for any corrupt/non-existent paths
IMG_DIR = f"{DATA_PATH}/images"
valid_images = set(os.listdir(IMG_DIR))

df = df[df["image_name"].isin(valid_images)].reset_index(drop=True)
print("Clean dataset size:", len(df))

Clean dataset size: 6992


In [40]:
from src.dataset import MemeDataset
from torch.utils.data import DataLoader

dataset = MemeDataset(
    df=df,
    img_dir=f"{DATA_PATH}/images",
    tokenizer=tokenizer,
    transform=transform
)

loader = DataLoader(dataset, batch_size=4, shuffle=True)

In [41]:
batch = next(iter(loader))

print(batch["input_ids"].shape)
print(batch["image"].shape)
print(batch["label"])

torch.Size([4, 128])
torch.Size([4, 3, 224, 224])
tensor([0, 0, 2, 1])


In [42]:
from src.model_text import TextModel

model = TextModel(num_classes=4)

batch = next(iter(loader))

outputs = model(
    input_ids=batch["input_ids"],
    attention_mask=batch["attention_mask"]
)

print(outputs.shape)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([4, 4])


In [43]:
print(outputs)

tensor([[-0.0803, -0.3676, -0.0379, -0.2584],
        [-0.3566, -0.2417, -0.1984, -0.1320],
        [-0.3887, -0.1453, -0.4008, -0.0253],
        [-0.1965,  0.0668, -0.1908,  0.1430]], grad_fn=<AddmmBackward0>)


In [44]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = torch.device("cpu")
print(device)

cuda


In [45]:
model = TextModel(num_classes=4)
model.to(device)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


TextModel(
  (bert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
            (lin1): Linear(in_fe

In [46]:
import torch.nn as nn
from torch.optim import AdamW

criterion = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=2e-5)

In [47]:
# # Testing src/train.py

# # run with kaggle T4 not with P100

# from src.train import train_one_epoch

# EPOCHS = 1  # small for now

# for epoch in range(EPOCHS):
#     loss = train_one_epoch(model, loader, optimizer, criterion, device)
#     print(f"Epoch {epoch+1} Loss: {loss:.4f}")

In [48]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["offensive"]
)

In [49]:
train_dataset = MemeDataset(
    df=train_df,
    img_dir="/kaggle/input/memotion-dataset/images",
    tokenizer=tokenizer,
    transform=transform
)

val_dataset = MemeDataset(
    df=val_df,
    img_dir="/kaggle/input/memotion-dataset/images",
    tokenizer=tokenizer,
    transform=transform
)

from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4)

In [58]:
from src.train import train_one_epoch, evaluate

EPOCHS = 2 # low for start

for epoch in range(EPOCHS):
    loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    acc, f1 = evaluate(model, val_loader, device)

    print(f"\nEpoch {epoch+1}")
    print(f"Loss: {loss:.4f}")
    print(f"Accuracy: {acc:.4f}")
    print(f"F1 Score: {f1:.4f}")

100%|██████████| 1399/1399 [01:02<00:00, 22.36it/s]



Epoch 1
Loss: 0.9997
Accuracy: 0.3738
F1 Score: 0.2970


100%|██████████| 1399/1399 [01:04<00:00, 21.81it/s]



Epoch 2
Loss: 0.9973
Accuracy: 0.3738
F1 Score: 0.2970


In [59]:
torch.save(model.state_dict(), "text_model.pt")

In [60]:
from src.model_text import TextModel

model = TextModel(num_classes=4)
model.load_state_dict(torch.load("text_model.pt"))
model.to(device)
model.eval()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


TextModel(
  (bert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
            (lin1): Linear(in_fe

In [61]:
# Predict Text Function and Test Predictions

def predict_text(text, model, tokenizer, device):
    model.eval()

    encoding = tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model(input_ids, attention_mask)
        pred = torch.argmax(outputs, dim=1).item()

    return pred

id2label = {
    0: "not_offensive",
    1: "slight",
    2: "very_offensive",
    3: "hateful_offensive"
}

text = "I hate you so much"

pred = predict_text(text, model, tokenizer, device)

print("Prediction:", id2label[pred])

Prediction: slight


In [66]:
from src.model_image import ImageModel

image_model = ImageModel(num_classes=4).to(device)

batch = next(iter(train_loader))

outputs = image_model(batch["image"].to(device))

print(outputs.shape)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 170MB/s] 


torch.Size([4, 4])


In [67]:
def train_image(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0

    for batch in loader:
        images = batch["image"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [68]:
import torch.nn as nn
from torch.optim import AdamW

criterion = nn.CrossEntropyLoss()
optimizer = AdamW(image_model.parameters(), lr=1e-4)

EPOCHS = 2

for epoch in range(EPOCHS):
    loss = train_image(image_model, train_loader, optimizer, criterion, device)
    acc, f1 = evaluate(image_model, val_loader, device)

    print(f"\nEpoch {epoch+1}")
    print(f"Loss: {loss:.4f}")
    print(f"Accuracy: {acc:.4f}")
    print(f"F1 Score: {f1:.4f}")

TypeError: ImageModel.forward() takes 2 positional arguments but 3 were given